In [1]:
import os
import sys

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv

load_dotenv()

# 가상환경 확인
print(f"Python 실행 경로: {sys.executable}")
if ".venv" in sys.executable:
    print("[OK] .venv 가상환경 사용 중")
else:
    print("[WARN] .venv 가상환경이 아닌 인터프리터가 선택되어 있습니다")

print("[OK] OpenAI API 키 설정됨" if os.getenv("OPENAI_API_KEY") else "[ERROR] OpenAI API 키 미설정")
print("[OK] Anthropic API 키 설정됨" if os.getenv("ANTHROPIC_API_KEY") else "[ERROR] Anthropic API 키 미설정")
print("[OK] Google API 키 설정됨" if os.getenv("GOOGLE_API_KEY") else "[ERROR] Google API 키 미설정")
print("[OK] LangSmith 키 설정됨" if os.getenv("LANGSMITH_API_KEY") else "[ERROR] LangSmith 키 미설정")

print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

Python 실행 경로: d:\yugyeong\hanwha_0902\days\ex_0929\.venv\Scripts\python.exe
[OK] .venv 가상환경 사용 중
[OK] OpenAI API 키 설정됨
[ERROR] Anthropic API 키 미설정
[OK] Google API 키 설정됨
[OK] LangSmith 키 설정됨
LangChain: 1.4.3
LangChain Core: 1.6.5
LangChain OpenAI: 1.6.6


In [4]:
import requests

json_data = requests.get("https://api.smith.langchain.com/openapi.json").json()

In [13]:
from langchain_text_splitters import RecursiveJsonSplitter

splitter = RecursiveJsonSplitter(max_chunk_size=300)

json_chunks = splitter.split_json(
    json_data = json_data
)

In [14]:
docs = splitter.create_documents(texts=[json_data])

texts = splitter.split_text(json_data = json_data)

print(docs[0].page_content)

print("===" * 20)

print(texts[0])

{"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n"}}
{"openapi": "3.1.0", "info": {"title": "LangSmith", "description": "The LangSmith API is used to programmatically create and manage LangSmith resources.\n\n## Host\nhttps://api.smith.langchain.com\n\n## Authentication\nTo authenticate with the LangSmith API, set the `X-Api-Key` header\nto a valid [LangSmith API key](https://docs.langchain.com/langsmith/create-account-api-key#create-an-api-key).\n\n"}}


In [17]:
print([len(text) for text in texts][:10])

print(texts[1])

[404, 264, 226, 214, 238, 344, 207, 227, 242, 231]
{"info": {"version": "0.1.0"}, "paths": {"/api/v1/info/health": {"get": {"tags": ["info"], "summary": "Get Health Info", "description": "Get health information about the current deployment of LangSmith.", "operationId": "get_health_info_api_v1_info_health_get"}}}}


In [26]:
import json

json_data = json.loads(texts[2])

json_data["paths"]

{'/api/v1/info/health': {'get': {'responses': {'200': {'description': 'Successful Response',
     'content': {'application/json': {'schema': {'$ref': '#/components/schemas/HealthInfoGetResponse'}}}}},
   'x-public': True}}}

In [30]:
texts = splitter.split_text(json_data=json_data, convert_lists=True)

docs[2]

Document(metadata={}, page_content='{"paths": {"/api/v1/info/health": {"get": {"responses": {"200": {"description": "Successful Response", "content": {"application/json": {"schema": {"$ref": "#/components/schemas/HealthInfoGetResponse"}}}}}, "x-public": true}}}}')

| 구분         | Splitter                                         | 핵심                |
| ---------- | ------------------------------------------------ | ----------------- |
| ① 문자       | `CharacterTextSplitter`                          | 구분자 기준            |
| ② 텍스트      | `RecursiveCharacterTextSplitter`                 | 여러 구분자로 단계적 분할    |
| ③ 토큰 고려    | `Recursive...from_tiktoken_encoder()`            | Recursive + 토큰 측정 |
| ④ 토큰 기준    | `TokenTextSplitter`                              | 토큰 수 기준           |
| ⑤ 의미 기반    | `SemanticChunker`                                | 의미 차이 기준          |
| ⑥ 코드       | `RecursiveCharacterTextSplitter.from_language()` | 언어별 코드 구조 기준      |
| ⑦ Markdown | `MarkdownHeaderTextSplitter`                     | Markdown 헤더 구조 기준 |
| ⑧ HTML     | `HTMLHeaderTextSplitter`                         | HTML 헤더 구조 기준     |
| ⑨ JSON     | `RecursiveJsonSplitter`                          | JSON 구조를 유지하며 분할  |
